# Building a Text-to-SQL Agent with DuckDB, MotherDuck and LangChain

This notebook demonstrates how to build a reliable Text-to-SQL agent that doesn't just "Claude-it" and hope for the best. Instead, it uses an iterative loop: inspecting the schema, drafting a query, running it, reading errors, and self-correcting.

### Why this stack?
- **DuckDB**: Fast, in-process columnar engine for analytical queries.
- **MotherDuck**: Cloud persistence and heavy lifting for DuckDB.
- **LangChain**: Scaffolding for tool-calling, prompt routing, and SQL toolkit.

## Setting Up the Environment

Install the required libraries. Note: `duckdb-engine` is essential for SQLAlchemy compatibility.

In [1]:
%pip install -U langchain langchain-community langchain-google-genai duckdb duckdb-engine sqlalchemy==2.0.34 python-dotenv


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Make sure you have a `.env` file with the following keys:
- `MOTHERDUCK_TOKEN`
- `GOOGLE_API_KEY`

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

# Load environment variables
load_dotenv()

# Connecting to MotherDuck's built-in sample data
# lazy_table_reflection=True prevents fetching all schemas at once
db = SQLDatabase.from_uri("duckdb:///md:sample_data", lazy_table_reflection=True)

print(f"Dialect: {db.dialect}")
print(f"Usable tables: {db.get_usable_table_names()[:5]}... (truncated)")

Attempting to automatically open the SSO authorization page in your default browser.
Please open this link to login into your account: https://auth.motherduck.com/activate?user_code=GGKQ-VCMP


Token successfully retrieved ✅

You can display the token and store it as an environment variable to avoid having to log in again:
  PRAGMA PRINT_MD_TOKEN;
Dialect: duckdb
Usable tables: ['ambient_air_quality', 'hacker_news', 'movies', 'rideshare', 'service_requests']... (truncated)


## Wiring Up the LLM and Toolkit

We use a strong reasoning model for production reliability. Gemini 1.5 Pro (or the newer preview) works well with LangChain's tool-calling interface.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# Note: Use the model best suited for your environment (e.g., gemini-1.5-pro or gemini-2.0-flash)
llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-pro-preview")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

## DuckDB-Specific System Prompt

Standard SQL prompts often fail with DuckDB's unique dialect. We must explicitly specify DuckDB idioms and functions.

In [4]:
duckdb_system_prompt = """You are an expert data analyst interacting with a {dialect} database.
Given an input question, create a syntactically correct {dialect} SQL query to run, then look at the results and return the answer.

DuckDB Specifics:
- Use DuckDB-specific functions where appropriate (e.g., EPOCH for scalar time extraction, STRFTIME for formatting).
- DuckDB supports reading directly from Parquet/CSVs, but assume tables exist unless told otherwise.
- Never use PostgreSQL-specific functions that do not exist in DuckDB.
- ALWAYS append LIMIT {top_k} to your queries unless you are aggregating data, to prevent pulling too many rows.

Only use the tables available to you. Do NOT hallucinate table names.
"""

In [5]:
prefix = duckdb_system_prompt.format(dialect=db.dialect, top_k=10)

## Creating the Agent

The `handle_parsing_errors=True` flag is vital for production—it allows the agent to self-correct if the LLM emits a poorly formatted response.

In [6]:
from langchain_community.agent_toolkits import create_sql_agent

agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type="tool-calling",
    agent_executor_kwargs={"handle_parsing_errors": True},
    prefix=prefix
)

## Testing the Agent

Try asking a question about the taxi data!

In [7]:
question = "What was the average tip amount broken down by passenger count in the NYC taxi data?"
agent_executor.invoke({"input": question})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded: [{'type': 'text', 'text': '  Then I should write a query to get the average tip amount broken down by passenger count.  Then I should check the query.  Then I should run the query.  Then I should return the answer.\n\nFirst, I will list the tables in the database.\n', 'index': 0}]

ambient_air_quality, hacker_news, movies, rideshare, service_requests, survey_results, survey_schemas, taxi
Invoking: `sql_db_schema` with `{'table_names': 'taxi'}`




/Users/as/Documents/venv/lib/python3.13/site-packages/duckdb_engine/__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(



CREATE TABLE taxi (
	"VendorID" BIGINT, 
	tpep_pickup_datetime TIMESTAMP WITHOUT TIME ZONE, 
	tpep_dropoff_datetime TIMESTAMP WITHOUT TIME ZONE, 
	passenger_count FLOAT, 
	trip_distance FLOAT, 
	"RatecodeID" FLOAT, 
	store_and_fwd_flag VARCHAR, 
	"PULocationID" BIGINT, 
	"DOLocationID" BIGINT, 
	payment_type BIGINT, 
	fare_amount FLOAT, 
	extra FLOAT, 
	mta_tax FLOAT, 
	tip_amount FLOAT, 
	tolls_amount FLOAT, 
	improvement_surcharge FLOAT, 
	total_amount FLOAT, 
	congestion_surcharge FLOAT, 
	airport_fee FLOAT
)

/*
3 rows from taxi table:
VendorID	tpep_pickup_datetime	tpep_dropoff_datetime	passenger_count	trip_distance	RatecodeID	store_and_fwd_flag	PULocationID	DOLocationID	payment_type	fare_amount	extra	mta_tax	tip_amount	tolls_amount	improvement_surcharge	total_amount	congestion_surcharge	airport_fee

*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT passenger_count, AVG(tip_amount) AS average_tip_amount FROM taxi GROUP BY passenger_count ORDER BY passenger_count;'}`


``

{'input': 'What was the average tip amount broken down by passenger count in the NYC taxi data?',
 'output': [{'type': 'text',
   'text': 'The average tip amount broken down by passenger count in the NYC taxi data is as follows:\n\n* **0 passengers:** $2.54\n* **1 passenger:** $2.78\n* **2 passengers:** $3.04\n* **3 passengers:** $2.86\n* **4 passengers:** $2.85\n* **5 passengers:** $2.77\n* **6 passengers:** $2.82\n* **7 passengers:** $7.17\n* **8 passengers:** $7.35\n* **9 passengers:** $6.22\n* **Unknown (missing passenger count):** $3.53',
   'index': 0,
   'extras': {'signature': 'EtgDCtUDAb4+9vupvYsBLtFmkJOBQl8CJMdah/73vjoEq2FCo/iQjEdcblY4izSmm0hpyE9YnvEgCXuvaSHnAWEkX2W8sWaRVUkLWSChaTv/X55OJ9hK7R4iw+1pEOyXZ9Iht7ct3PNrCtqBjxtMJ6RdGYkiDpdSD48tVhVmx/NP0WCS+yx46ufajzkckQj/+piO9WN1GOsLXgZewk12zb6qwlpP3a/OyxILtQr2g8QMugaRR1PLjuYqoSXs0N0E86HFL3uRcQ8oaduQQy3rShyjSCP0VhF6JOtPDTYIi1vNUlXhJJtrG2LE98r55+1mhaWYEkry/DZp7XWHxYN2fwpDAfqqogeWPxoC4SJ3gXXubQKnhRQvC2ZKmierTGZiFtjb1fTKAWdZneFRHy1VsRm

## Production Considerations

1.  **Security**: Use read-only database connections. SQL agents are vulnerable to prompt injection.
2.  **Token Usage**: Always enforce `LIMIT` to avoid overflowing context windows.
3.  **Semantic Layer**: Inject business logic (e.g., how "Total Cost" is calculated) into the system prompt.
4.  **Human-in-the-Loop**: For complex workloads, use LangGraph to add an approval step before SQL execution.